# TP3 - Recomendação de Disciplinas e Trilhas de Carreira (UFMG)

**Disciplina:** Fundamentos de Inteligência Artificial  
**Tarefa primária de NLP:** busca semântica com Sentence-BERT  
**Objetivo:** transformar uma descrição livre de objetivos profissionais em carreiras sugeridas, disciplinas aderentes e um roadmap que respeita pré-requisitos.

Este notebook é a versão reprodutível do projeto. Ele carrega os dados curados, executa o pipeline completo, roda a avaliação quantitativa e aponta os principais erros observados.

## 1. Como usar no Colab

Se estiver no Google Colab, primeiro clone o repositório e entre na pasta do projeto. Localmente, basta abrir este notebook na raiz do repositório.

```bash
git clone https://github.com/erikrocha98/sentence-bert-career-recommender.git
cd sentence-bert-career-recommender
pip install -r requirements.txt
```

A primeira execução pode baixar o modelo `paraphrase-multilingual-MiniLM-L12-v2` do Hugging Face.

In [ ]:
# No Colab, descomente se ainda não instalou as dependências.
# !pip install -r requirements.txt

## 2. Carregamento e inspeção dos dados

A base combina duas fontes curadas:

- **Fonte A - disciplinas:** catálogo curado de 30 disciplinas a partir da estrutura curricular disponível no repositório; cada registro possui nome, ementa, objetivos, carga horária e pré-requisitos estruturados como códigos.
- **Fonte C - perfis de carreira:** descrição, competências, tecnologias e área.

Os pré-requisitos permitem montar uma trilha por semestre depois que as disciplinas são ranqueadas. Os arquivos `*_emb.pkl` são artefatos derivados desses JSONs, não fontes independentes de dados.

In [ ]:
import json
from collections import Counter
from pathlib import Path

ROOT = Path('.')
disciplinas = json.loads((ROOT / 'data' / 'disciplinas.json').read_text(encoding='utf-8'))
perfis = json.loads((ROOT / 'data' / 'perfis.json').read_text(encoding='utf-8'))

print(f'Disciplinas: {len(disciplinas)}')
print(f'Perfis de carreira: {len(perfis)}')
print(f'Arestas de pré-requisito: {sum(len(d["prerequisitos"]) for d in disciplinas)}')
print('Perfis por área:')
for area, n in Counter(p['area'] for p in perfis).most_common():
    print(f'  {area}: {n}')

## 3. Validação dos contratos de dados

Antes de rodar o modelo, validamos se os registros possuem os campos obrigatórios e se todos os pré-requisitos apontam para códigos existentes no catálogo.

In [ ]:
from src.schema import validar_disciplina, validar_perfil

for d in disciplinas:
    validar_disciplina(d)
for p in perfis:
    validar_perfil(p)

codigos = {d['codigo'] for d in disciplinas}
prereq_faltando = sorted({p for d in disciplinas for p in d['prerequisitos']} - codigos)
print('Pré-requisitos inexistentes:', prereq_faltando)
assert not prereq_faltando
print('Dados válidos.')

## 4. Metodologia: pipeline em dois saltos

O sistema usa embeddings de Sentence-BERT e similaridade de cosseno.

1. **Hop 1 - aluno → carreira:** a consulta do aluno é comparada com os perfis de carreira.
2. **Hop 2 - carreira → disciplinas:** as competências da carreira escolhida viram uma nova consulta contra as ementas das disciplinas.
3. **Roadmap:** as disciplinas recomendadas são ordenadas respeitando pré-requisitos.

Os embeddings das bases ficam pré-computados em `data/perfis_emb.pkl` e `data/disciplinas_emb.pkl`; apenas a consulta do usuário é vetorizada em tempo de uso.

In [ ]:
from src.busca import carregar_colecao
from src.pipeline import recomendar_trilha

perfis_emb = carregar_colecao(ROOT / 'data' / 'perfis_emb.pkl')
disciplinas_emb = carregar_colecao(ROOT / 'data' / 'disciplinas_emb.pkl')

print(len(perfis_emb), 'embeddings de perfis')
print(len(disciplinas_emb), 'embeddings de disciplinas')

## 5. Demonstração do pipeline completo

A consulta abaixo simula um aluno descrevendo seus interesses. O resultado inclui carreira, disciplinas, roadmap e justificativa.

In [ ]:
query = 'Quero trabalhar com IA generativa, processamento de linguagem natural e busca semântica'
resultado = recomendar_trilha(query, perfis, perfis_emb, disciplinas, disciplinas_emb, top_k=5)

nomes_disc = {d['codigo']: d['nome'] for d in disciplinas}
print('Consulta:', query)
print('Carreira:', resultado['carreira'], f"score={resultado['score']:.3f}")
print('\nDisciplinas recomendadas:')
for codigo, score in resultado['disciplinas']:
    print(f'  {score:.3f}  {codigo}  {nomes_disc[codigo]}')

print('\nRoadmap:')
for semestre, itens in resultado['roadmap'].items():
    print(f'{semestre}º semestre')
    for item in itens:
        marcador = ' [pré-requisito]' if item['origem'] == 'prerequisito' else ''
        print(f"  - {item['codigo']} {nomes_disc[item['codigo']]}{marcador}")

print('\nJustificativa:', resultado['justificativa'])

## 6. Avaliação quantitativa

O projeto compara Sentence-BERT contra um baseline TF-IDF. A avaliação usa:

- 20 consultas de carreira com uma carreira esperada;
- 13 consultas de competências com três disciplinas esperadas cada;
- métricas top-1, hit@3, Precision@3, Recall@3 e MRR.

Os rótulos esperados só são usados depois do ranqueamento. Não há treino supervisionado, portanto não há vazamento treino-teste.

In [ ]:
%run scripts/avaliar_modelos.py

## 7. Leitura das métricas geradas

O script salva os resultados em `results/avaliacao_metricas.json` e gera figuras para o relatório.

In [ ]:
metricas = json.loads((ROOT / 'results' / 'avaliacao_metricas.json').read_text(encoding='utf-8'))
metricas

## 8. Análise de erros

A célula abaixo lista casos em que o top-1 de carreira falhou ou em que a Precision@3 das disciplinas ficou abaixo de 1.0. Esses exemplos alimentam a seção crítica do relatório.

In [ ]:
import csv

print('Erros ou ambiguidades no Hop 1:')
with open(ROOT / 'results' / 'avaliacao_carreiras.csv', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        if row['acerto_top1'] == '0':
            print('-', row['metodo'], '| esperado:', row['esperado'], '| top1:', row['top1'])

print('\nCasos imperfeitos no Hop 2:')
with open(ROOT / 'results' / 'avaliacao_disciplinas.csv', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        if float(row['precision_at_3']) < 1.0:
            print('-', row['metodo'], '| P@3:', row['precision_at_3'], '| top3:', row['top3'])

## 9. Figuras do relatório

As figuras abaixo são geradas automaticamente pelo script de avaliação e usadas no relatório LaTeX.

In [ ]:
from IPython.display import Image, display

for nome in [
    'distribuicao_perfis_area.png',
    'comparacao_metricas.png',
    'similaridade_queries_carreiras.png',
]:
    print(nome)
    display(Image(filename=str(ROOT / 'results' / 'figures' / nome)))

## 10. Interface opcional

A demonstração web usa Gradio. Localmente, rode `python app.py`. No Colab, pode-se adaptar o final de `app.py` para `demo.launch(share=True)` e gerar um link temporário.